Eric Arnold

ML Midterm

Prof. Khoei

In [15]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.model_selection import train_test_split
from pprint import pprint

from scipy.stats import chi2_contingency
from collections import defaultdict

## Load datasets

When loading, polars did not automatically recognize any of the dtypes, so I had to manually cast them. Some categories in the Assess portion of the dataset are actually floats, but they were very large, so rounding introduced by casting them as integers is small.

In [16]:
cols = ['ID', 'CENTER', 'FY', 'SIC', 'NAICS', 'STATE', 'SALES', 'EMPLOYEES', 'PLANT_AREA', 'PRODUCTS', 'PRODUNITS', 'PRODLEVEL', 'PRODHOURS', 'RECC_COUNT_recc', 'EC_plant_cost', 'EC_plant_usage', 'ED_plant_cost', 'ED_plant_usage', 'EF_plant_cost', 'E2_plant_cost', 'E2_plant_usage', 'E3_plant_cost', 'E3_plant_usage', 'E4_plant_cost', 'E4_plant_usage', 'E5_plant_cost', 'E5_plant_usage', 'E6_plant_cost', 'E6_plant_usage', 'E7_plant_cost', 'E7_plant_usage', 'E8_plant_cost', 'E8_plant_usage', 'E9_plant_cost', 'E9_plant_usage', 'E10_plant_cost', 'E10_plant_usage', 'E11_plant_cost', 'E11_plant_usage', 'E12_plant_cost', 'E12_plant_usage', 'W0_plant_cost', 'W0_plant_usage', 'W1_plant_cost', 'W1_plant_usage', 'W2_plant_cost', 'W2_plant_usage', 'W3_plant_cost', 'W3_plant_usage', 'W4_plant_cost', 'W4_plant_usage', 'W5_plant_cost', 'W5_plant_usage', 'W6_plant_cost', 'W6_plant_usage']
str_cols = ["ID", "CENTER", "STATE", "PRODUCTS"] 
int_cols = [c for c in cols if c not in str_cols]
schema = {c: pl.Utf8 for c in str_cols} | {c: pl.Int64 for c in int_cols}

ASSESS = pl.read_excel("ITAC_Database_20261002.xlsx", sheet_name="ASSESS", schema_overrides=schema)

cols = ['SUPERID', 'ID', 'ARC2', 'IMPSTATUS', 'IMPCOST', 'PSOURCCODE', 'PCONSERVED', 'PSOURCONSV', 'PSAVED', 'SSOURCCODE', 'SCONSERVED', 'SSOURCONSV', 'SSAVED', 'TSOURCCODE', 'TCONSERVED', 'TSOURCONSV', 'TSAVED', 'QSOURCCODE', 'QCONSERVED', 'QSOURCONSV', 'QSAVED', 'REBATE', 'INCREMNTAL', 'FY', 'IC_CAPITAL', 'IC_OTHER', 'PAYBACK', 'BPTOOL']
str_cols = ["SUPERID", "ID", "AR_NUMBER", "IMPSTATUS", "BPTOOL", "PSOURCCODE", "SSOURCCODE", "TSOURCCODE", "QSOURCCODE", "REBATE", "INCREMNTAL"]
float_cols = ["ARC2", "PAYBACK"]
int_cols = ["APPCODE", "AR_NUMBER", "FY", "IMPCOST", "PCONSERVED", "PSOURCONSV", "PSAVED", "SCONSERVED", "SSOURCONSV", "SSAVED", "TCONSERVED", "TSOURCONSV", "TSAVED", "QCONSERVED", "QSOURCONSV", "QSAVED", "IC_CAPITAL", "IC_OTHER"]
schema = ({c: pl.Utf8 for c in str_cols} | {c: pl.Float64 for c in float_cols} | {c: pl.Int64 for c in int_cols})


RECC = pl.read_excel("ITAC_Database_20261002.xlsx", sheet_name="RECC", schema_overrides=schema)

# Problem Statement

The Industrial Training and Assessment Centers (ITAC) organization is a DOE program that assesses industrial facilities, chiefly small and medium-sized factories, and makes recommendations about how to save energy and money on production, roughly speaking. The dataset indicates whether or not the recommended changes were implemented, how much it cost, and how much was saved (in terms of resource units, which might be gallons of fuel, kilowatts of electricity, etc.).

The obvious thing to comb through all the variables and see if any are associated with implementation of recommendations, and subsequent savings. 

In [17]:
ASSESS_ids = ASSESS["ID"].unique()
RECC_ids = RECC["ID"].unique()
null_recc_ids = RECC_ids.filter(~RECC_ids.is_in(ASSESS_ids.implode()))
null_assess_ids = ASSESS_ids.filter(~ASSESS_ids.is_in(RECC_ids.implode()))
print(null_recc_ids)
print(null_assess_ids)
RECC = RECC.filter(~pl.col("ID").is_in(null_recc_ids.implode()))
# fig = go.Figure(go.Histogram(RECC[id]))

shape: (6,)
Series: 'ID' [str]
[
	"UU0309"
	"UA0312"
	"TR0076"
	"WV0702"
	"IC0353"
	"GT1011"
]
shape: (0,)
Series: 'ID' [str]
[
]


In [18]:
assessments_per = RECC["ID"].value_counts()["count"].to_list()
implementations_per = RECC.filter(pl.col("IMPSTATUS") == "I")["ID"].value_counts()["count"].to_list()
negs_per = RECC.filter(pl.col("IMPSTATUS") == "N")["ID"].value_counts()["count"].to_list()
state_df = RECC.select("ID", "IMPSTATUS").join(ASSESS.select("ID", "STATE"), on="ID", how="left")
state_df = state_df.group_by("ID", "IMPSTATUS", "STATE").agg(pl.len()).sort("len")
state_assess = state_df.group_by("STATE").agg(pl.col("len").sum()).sort("len")
state_means = state_df.group_by("STATE").agg(pl.col("len").filter(pl.col("IMPSTATUS") == "I").sum() / pl.col("len").sum()).sort(by="len", descending=True)
subs = make_subplots(3, 2, subplot_titles=["Assessments per plant", "Implementations per plant", "Negatives per plant", "Assessments per state", "Mean Implementations/Assessments per plant per state"])

subs.add_trace(go.Histogram(x=assessments_per), 1, 1)
subs.add_trace(go.Histogram(x=implementations_per), 1, 2)
subs.add_trace(go.Histogram(x=negs_per), 2, 1)
subs.add_trace(go.Bar(x=state_assess["STATE"], y=state_assess["len"]), 2, 2)
subs.add_trace(go.Bar(x=state_means["STATE"], y=state_means["len"]), 3, 1)

subs.update_xaxes(title_text="Assessments", row=1, col=1)
subs.update_yaxes(title_text="Plants", row=1, col=1)
subs.update_xaxes(title_text="Implementations", row=1, col=2)
subs.update_yaxes(title_text="Plants", row=1, col=2)
subs.update_xaxes(title_text="Negatives", row=2, col=1)
subs.update_yaxes(title_text="Plants", row=2, col=1)
subs.update_xaxes(title_text="State", row=2, col=2)
subs.update_yaxes(title_text="Assessments", row=2, col=2)
subs.update_xaxes(title_text="State", row=3, col=1)
subs.update_yaxes(title_text="Mean Value", row=3, col=1)
subs.update_layout(height=768, width=1024)

subs.show()

The distribution of implementations per plant is slightly right-shifted compared to negatives per plant, indicating that recommendations are implemented more often than not. There is a huge amount of variability in terms of the ratio of successful implementations to total implementations per state, and in the number of assessments per state. For splitting the dataset into train/val, we probably want to balance our subset by state.

My strategy here is to select the ids and then apply them to the dataframes before I use them, instead of selecting rows.

In [19]:
ids = ASSESS.select("ID", "STATE").unique(subset="ID", keep="first", maintain_order=True)
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=0, stratify=ids["STATE"])

## Production Unit (produnit) analysis

IAC defines the unit type for measuring the quantity of a product produced using the following codes:

- 0: N/A
- 1: Pieces
- 2: Pounds
- 3: Tons
- 4: BBL (beer barrel?)
- 5: Thousands of Gallons
- 6: Thousands of Feet or Square Feet
- 7: Bushels

I wanted to verify that there was no significant difference between produnit proportions between train and val as a proxy for "the kind of thing produced." This is because my split did not take into account any information about what kinds of products were produced. Since states might vary greatly in terms of industry type, I wanted to test for this. There were no significant differences subdividing by state or overall.

In [54]:
pu_train = ASSESS.select("ID", "PRODUNITS", "STATE").filter(pl.col("ID").is_in(train_ids["ID"].implode()))
pu_train = pu_train.with_columns(pl.col("PRODUNITS").fill_null(0))
pu_state_train = pu_train.group_by("STATE", "PRODUNITS").agg(pl.len()).sort("STATE", "PRODUNITS")

pu_val = ASSESS.select("ID", "PRODUNITS", "STATE").filter(pl.col("ID").is_in(val_ids["ID"].implode()))
pu_val = pu_val.with_columns(pl.col("PRODUNITS").fill_null(0))
pu_state_val = pu_val.group_by("STATE", "PRODUNITS").agg(pl.len()).sort("STATE", "PRODUNITS")

pu_train = pu_train.with_columns((pl.col("STATE").cast(pl.String) + "_" + pl.col("PRODUNITS").cast(pl.String)).alias("sp"))
pu_val = pu_val.with_columns((pl.col("STATE").cast(pl.String) + "_" + pl.col("PRODUNITS").cast(pl.String)).alias("sp"))

print("Testing STATE x PRODUNITS x Split")
tbl = (
    pl.concat([
        pu_train.group_by("sp").agg(pl.len().alias("n")).with_columns(pl.lit("train").alias("split")),
        pu_val.group_by("sp").agg(pl.len().alias("n")).with_columns(pl.lit("val").alias("split")),
    ])
    .pivot(on="sp", index="split", values="n")
    .fill_null(0)
    .sort("split")
)
chi2, p, dof, expected = chi2_contingency(tbl.drop("split").to_numpy())
print("test_stat:", chi2, "p:", p, "dof:", dof)

print("\nTesting PRODUNITS x Split")
both = pl.concat([pu_state_train.with_columns(pl.lit("train").alias("split")),
                  pu_state_val.with_columns(pl.lit("val").alias("split"))])
overall = (
    both.group_by("split", "PRODUNITS").agg(pl.col("len").sum())
    .pivot(on="PRODUNITS", index="split", values="len")
    .fill_null(0)
    .sort("split")
)
chi2, p, dof, expected = chi2_contingency(overall.drop("split").to_numpy())
print("test_stat:", chi2, "p:", p, "dof:", dof)

Testing STATE x PRODUNITS x Split
test_stat: 287.24239379725435 p: 0.976922289493944 dof: 337

Testing PRODUNITS x Split
test_stat: 3.7126849410443343 p: 0.8122116540708135 dof: 7


Now that the dataset is split, we can do more EDA on train only.

Here we can examine the relationships between the assessment variables. Since these are upstream from the recommendation, we can treat them as indepdent variables. I'm analyzing these separately from the resource utilization variables since they're relatively dense.

Log transforming the distributions for sales, employees, plant area, prod level, and recc count normalizes these distributions.

Nulls in employees, sales, and prod level seem like good candidates for interpolation within their respective categories (state, produnit). About 1/4 of the data lacks a plant area measurement, and I'd expect that this is associated with agriculture or some industry where this is hard to quanitfy.

For prod hours (annual), a large number of entries have 8600-8700 hours, which across 365 days reflects 24 hours a day of production. I think that anything more than 24/hours a day doesn't make sense, so we can cap these at the maximum possible value. For interpolation, we can probably take the mean of its associate category, depending on how prodhours is assocaited with other variables.

In [55]:
ASSESS_train = ASSESS.filter(pl.col("ID").is_in(train_ids["ID"].implode()))
a_cols = ['SALES', 'EMPLOYEES', 'PLANT_AREA', 'PRODLEVEL', 'PRODHOURS', 'RECC_COUNT_recc']
nulls = {cat: ASSESS_train[cat].is_null().sum() + (ASSESS_train[cat] == 0).sum() for cat in a_cols}
pprint(nulls)

sales_trf = ASSESS_train["SALES"].log().to_numpy()
empl_trf = ASSESS_train["EMPLOYEES"].log().to_numpy()
area_trf = ASSESS_train["PLANT_AREA"].log().to_numpy()
lvl_trf = ASSESS_train["PRODLEVEL"].log().to_numpy()
hrs_trf = (ASSESS_train.with_columns(pl.col("PRODHOURS").clip(upper_bound=8760))["PRODHOURS"] / 365).to_numpy()
recc_trf = ASSESS_train["RECC_COUNT_recc"].log().to_numpy()

subs = make_subplots(2, 3, subplot_titles=[f"Distribution of {cat}" for cat in a_cols])
subs.add_trace(go.Histogram(x=sales_trf), 1, 1)
subs.add_trace(go.Histogram(x=empl_trf), 1, 2)
subs.add_trace(go.Histogram(x=area_trf), 1, 3)
subs.add_trace(go.Histogram(x=lvl_trf), 2, 1)
subs.add_trace(go.Histogram(x=hrs_trf), 2, 2)
subs.add_trace(go.Histogram(x=recc_trf), 2, 3)

subs.update_xaxes(title_text="Sales $$ (log scale)", row=1, col=1)
subs.update_yaxes(title_text="Plants", row=1, col=1)
subs.update_xaxes(title_text="Employees (log scale)", row=1, col=2)
subs.update_yaxes(title_text="Plants", row=1, col=2)
subs.update_xaxes(title_text="Plant Area  (log scale)", row=1, col=3)
subs.update_yaxes(title_text="Plants", row=1, col=3)
subs.update_xaxes(title_text="Production Level (log scale)", row=2, col=1)
subs.update_yaxes(title_text="Plants", row=2, col=1)
subs.update_xaxes(title_text="Production Hours / Day", row=2, col=2)
subs.update_yaxes(title_text="Plants", row=2, col=2)
subs.update_xaxes(title_text="Recommendation Count (log scale)", row=2, col=3)
subs.update_yaxes(title_text="Plants", row=2, col=3)

subs.update_layout(height=768, width=1024, showlegend=False)

subs.show()

{'EMPLOYEES': 13,
 'PLANT_AREA': 3262,
 'PRODHOURS': 67,
 'PRODLEVEL': 1013,
 'RECC_COUNT_recc': 0,
 'SALES': 102}


## Correlation Matrix

I chose to use Pearson here because the transformations were mostly successful at normalizing the data, and it is more powerful than Spearman. A large enough amount of the data is non-null for inference on these variables to be robust.

Correlations between the variables are unexpectedly low, which throws a wrench into the interpolation method. Interestingly there is no correlation with year (FY) and sales, which I expected to be related due to inflation, and the fact that the economy is continuously "growing."

In [66]:
corr = pl.DataFrame({"SALES": sales_trf, "EMPLOYEES": empl_trf, "PLANT_AREA": area_trf, "PRODLEVEL": lvl_trf, "PRODHOURS": hrs_trf, "RECC_COUNT_recc": recc_trf, "FY": ASSESS_train["FY"]
}).with_columns(pl.when(pl.all().is_finite()).then(pl.all()).otherwise(None))
print("shape before dropping nulls", corr.shape)
corr_full = corr.drop_nulls()
print("after", corr_full.shape)

corells = corr_full.corr()
fig = go.Figure(go.Heatmap(z = corells.to_numpy(), x = corells.columns, y = corells.columns, zmin=-1, zmax=1, text=np.round(corells.to_numpy(), 2), texttemplate="%{text}"))
fig.update_layout(height=768, width=768, title_text="Correlation Matrix Dense Assessment Variables")
fig.show()

shape before dropping nulls (18379, 7)
after (14486, 7)


## Sparse Assessment Data

This category of the dataset deals with resource consumption, such as electrical consumption, steam consumption, natural gas consumption, etc., and waste generation. I approached this set of variables separately because most plants contain only a subset of resources utilized and wastes generated. My plan for these variables is to treat zeros as zeros and use either PCA or UMAP to do dimensionality reduction. I expect UMAP to handle sparsity better since it uses euclidean distance and does not consider directions of variance individually (as a linear combination), but I'll use PCA to see about how many dimensions reside in the data.